In [7]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_FOLDER = Path("../../data")
INPUT_PATH = DATA_FOLDER / "processed" / "processed_data.csv"
OUTPUT_FOLDER = DATA_FOLDER / "features" / "tfidf"

MATRIX_PATH = OUTPUT_FOLDER / "tfidf_matrix.npz"
VECTORIZER_PATH = OUTPUT_FOLDER / "tfidf_vectorizer.joblib"
DOCUMENTS_PATH = OUTPUT_FOLDER / "tfidf_documents.csv"
VOCABULARY_PATH = OUTPUT_FOLDER / "tfidf_vocabulary.csv"

print(f"Input: {INPUT_PATH.resolve()}")
print(f"Output folder: {OUTPUT_FOLDER.resolve()}")

Input: C:\Works\uts-ir\src\data\processed\processed_data.csv
Output folder: C:\Works\uts-ir\src\data\features\tfidf


In [8]:
# Load Dataset

df = pd.read_csv(INPUT_PATH)

df.head()

,id,title,abstract,title_translated,abstract_translated,processed_token_text,processed_text
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,"['hubung', 'promosi', 'pustaka', 'majalah', 'd...",hubung promosi pustaka majalah dinding minat k...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,"['hubung', 'guna', 'search', 'features', 'jour...",hubung guna search features journals penuh but...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,"['hubung', 'sedia', 'fasilitas', 'pustaka', 'm...",hubung sedia fasilitas pustaka minat kunjung s...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,"['pengaruh', 'kerja', 'layan', 'prima', 'cente...",pengaruh kerja layan prima center of informati...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,"['kontribusi', 'literasi', 'informasi', 'mahas...",kontribusi literasi informasi mahasiswa proses...


# Weighting TF-IDF

In [9]:
vectorizer = TfidfVectorizer()

X_main = vectorizer.fit_transform(df["processed_text"])

print(f"Shape matriks TF-IDF (dokumen, term): {X_main.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Jumlah dokumen: {X_main.shape[0]:,}")
print(f"Jumlah term unik: {X_main.shape[1]:,}")

Shape matriks TF-IDF (dokumen, term): (5871, 21808)
Vocabulary size: 21,808
Jumlah dokumen: 5,871
Jumlah term unik: 21,808


In [10]:
n_rows, n_cols = X_main.shape
total_cells = n_rows * n_cols
n_nonzero = int(X_main.getnnz())
density_percent = 100 * (n_nonzero / total_cells) if total_cells else 0
sparsity_percent = 100 * (1 - (n_nonzero / total_cells)) if total_cells else 0

matrix_summary = pd.DataFrame({
    "metric": [
        "documents (rows)",
        "terms (columns)",
        "non-zero TF-IDF values",
        "total possible cells",
        "density (%)",
        "sparsity (%)",
    ],
    "value": [
        f"{n_rows:,}",
        f"{n_cols:,}",
        f"{n_nonzero:,}",
        f"{total_cells:,}",
        f"{density_percent:.2f}%",
        f"{sparsity_percent:.2f}%",
    ],
})

display(matrix_summary)

,metric,value
0,documents (rows),"5,871"
1,terms (columns),"21,808"
2,non-zero TF-IDF values,"358,465"
3,total possible cells,"128,034,768"
4,density (%),0.28%
5,sparsity (%),99.72%


In [11]:
# Ambil nama fitur sesuai urutan kolom pada matriks TF-IDF
feature_names = vectorizer.get_feature_names_out()

vocabulary_output = pd.DataFrame({
    "feature_index": np.arange(len(feature_names), dtype=int),
    "term": feature_names,
    "idf": vectorizer.idf_,
})

print("20 term dengan IDF tertinggi:")
display(
    vocabulary_output
    .nlargest(20, "idf")
    .sort_values("idf", ascending=False)
)

print("20 term dengan IDF terendah:")
display(
    vocabulary_output
    .nsmallest(20, "idf")
    .sort_values("idf", ascending=True)
)


20 term dengan IDF tertinggi:


,feature_index,term,idf
0,0,aa,8.984803
1,1,aaa,8.984803
3,3,aadministrasi,8.984803
4,4,aam,8.984803
6,6,aaspek,8.984803
8,8,aba,8.984803
10,10,abadi,8.984803
12,12,abang,8.984803
14,14,abbasyiyah,8.984803
15,15,abc,8.984803


20 term dengan IDF terendah:


,feature_index,term,idf
19653,19653,teliti,1.230322
6756,6756,hasil,1.290411
16032,16032,pustaka,1.324689
20430,20430,tuju,1.392437
12336,12336,metode,1.500154
3783,3783,data,1.556767
7532,7532,informasi,1.592156
9984,9984,kualitatif,1.872884
20103,20103,tingkat,1.891814
719,719,analisis,1.947337


# Export TF-IDF Results


In [12]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

save_npz(MATRIX_PATH, X_main, compressed=True)
joblib.dump(vectorizer, VECTORIZER_PATH)
df.to_csv(DOCUMENTS_PATH, index=False, encoding="utf-8-sig")

vocabulary_output.to_csv(VOCABULARY_PATH, index=False, encoding="utf-8-sig")

for path in [MATRIX_PATH, VECTORIZER_PATH, DOCUMENTS_PATH, VOCABULARY_PATH]:
    print(f"Saved: {path.resolve()} ({path.stat().st_size / (1024 * 1024):.2f} MB)")

Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_matrix.npz (3.04 MB)
Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_vectorizer.joblib (0.46 MB)
Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_documents.csv (27.39 MB)
Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_vocabulary.csv (0.70 MB)
